<a href="https://colab.research.google.com/github/jaylen11506/HP-Benchmarking/blob/main/Docker0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# Colab stand-in for docker/bench.Dockerfile. One place to change versions.
LLAMA_CPP_TAG = "b11170"# must match the Dockerfile
CUDA_ARCH = "75"# T4
BUILD_INFO = f"llama.cpp={LLAMA_CPP_TAG} GGML_CUDA=ON CUDA_ARCH={CUDA_ARCH}"# same format as Docker

import os, subprocess
from google.colab import drive

#!pip install -q fsspec==2025.12.0

# 0. Fail fast if this isn't a GPU runtime (Runtime > Change runtime type > T4 GPU).
if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
  raise SystemExit("No GPU. Switch the runtime to T4 GPU and rerun.")

drive.mount("/content/drive")
ROOT= "/content/drive/MyDrive/nano-bench"
CACHE = f"{ROOT}/llama-{LLAMA_CPP_TAG}-sm{CUDA_ARCH}"

# 1. Only system package Colab may be missing (llama.cpp needs it to build).
!apt-get -qq update && apt-get -qq install -y libcurl4-openssl-dev

# 2. Build the three programs the harness calls, once, then reuse from Drive.
#    Static build (BUILD_SHARED_LIBS=OFF) so the binaries still work after copying.
if not os.path.exists(f"{CACHE}/llama-bench"):
  !rm -rf /content/llama.cpp
  !git clone --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
  !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH} -DBUILD_SHARED_LIBS=OFF
  !cmake --build /content/llama.cpp/build -j$(nproc) --target llama-bench llama-server llama-cli
  !mkdir -p {CACHE} && cp /content/llama.cpp/build/bin/llama-* {CACHE}/
  with open(f"{CACHE}/BUILD_INFO", "w") as f:
    f.write(BUILD_INFO + "\n")
!chmod +x {CACHE}/llama-*
os.environ["PATH"] = f"{CACHE}:" + os.environ["PATH"]
os.environ["BUILD_INFO_PATH"] = f"{CACHE}/BUILD_INFO" # harness reads this instead of /opt/BUILD_INFO

# 3. Same Python environment as Docker: install from the shared requirements file.

!git clone -q https://github.com/jaylen11506/HP-Benchmarking.git /content/repo
%cd /content/repo
!pip install -q -r requirements-bench.txt

# 4. Results go to Drive so they survive disconnects.
os.environ["RESULTS_DIR"] = f"{ROOT}/results"
os.makedirs(os.environ["RESULTS_DIR"], exist_ok=True)

# 5. Record what this session actually got.
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!llama-bench --version

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo
name, compute_cap, driver_version
Tesla T4, 7.5, 580.82.07
version: 0.5.0-dev (build 1, commit a02c7f5)
built with GNU 13.3.0 for Linux x86_64


In [10]:
!ls /content/repo

ARM_NOTES.md  Dockerfile  README.md  results.csv  results_schema.json


In [6]:
!ls /content/drive/MyDrive/nano-bench/llama-b11170-sm75/

BUILD_INFO  llama-bench  llama-cli  llama-server


In [13]:
!git -C /content/repo pull
!ls /content/repo

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.26 KiB | 1.26 MiB/s, done.
From https://github.com/jaylen11506/HP-Benchmarking
   19d3d3f..694f585  main       -> origin/main
Updating 19d3d3f..694f585
Fast-forward
 requirements-bench.txt | 12 +++++++-----
 1 file changed, 7 insertions(+), 5 deletions(-)
ARM_NOTES.md  README.md		      results.csv
Dockerfile    requirements-bench.txt  results_schema.json
